# 035 · Nesterov Accelerated Gradient (NAG)

One change from momentum: measure the gradient at the point momentum is **about
to move you to**, not the point you are standing on.

$$w_{\text{look}} = w_t - \eta\beta v_{t-1}, \qquad v_t = \beta v_{t-1} + \nabla L(w_{\text{look}})$$

| Part | What we reproduce |
|---|---|
| A | the one-line difference, side by side |
| B | overshoot in 1-D: momentum **−6.04**, NAG **−3.55** — about **41% less** |
| C | the ravine: momentum reaches loss < 0.1 at **step 35**, NAG at **step 14** |
| D | why NAG's *final* loss can be higher while it arrives far sooner |

Needs `numpy`.

In [ ]:
import numpy as np

A, B, LR, STEPS = 1.0, 20.0, 0.02, 60

def grad(p):
    return np.array([A * p[0], B * p[1]])

def loss(p):
    return 0.5 * (A * p[0] ** 2 + B * p[1] ** 2)

## Part A — The one line that differs

In [ ]:
def descend(momentum, nesterov, steps=STEPS, lr=LR):
    p, v, path = np.array([9.0, 1.0]), np.zeros(2), []
    for _ in range(steps):
        path.append(p.copy())
        if nesterov:
            look = p - lr * momentum * v     # where momentum is about to put us
            v = momentum * v + grad(look)    # measure the gradient THERE
        else:
            v = momentum * v + grad(p)       # measure it HERE
        p = p - lr * v
    return path


print("Only the evaluation point changes. No new hyperparameter, no extra")
print("memory, and one extra gradient evaluation at a point you already know.")

## Part B — Overshoot, isolated

Strip the problem down to one dimension, `L = 0.5w²` from `w = 10`, so the only
thing left to measure is how far past zero each method swings.

In [ ]:
def bowl(momentum, nesterov, lr=0.1, steps=45, w0=10.0):
    w, v, path = w0, 0.0, []
    for _ in range(steps):
        path.append(w)
        look = w - lr * momentum * v if nesterov else w
        v = momentum * v + look
        w = w - lr * v
    return np.array(path)


mom, nag = bowl(0.9, False), bowl(0.9, True)
print(f"  momentum overshoots to {mom.min():+.3f}")
print(f"  NAG      overshoots to {nag.min():+.3f}")
print(f"\n  -> {(1 - nag.min() / mom.min()):.0%} less overshoot")

assert abs(mom.min() - (-6.037)) < 0.01
assert abs(nag.min() - (-3.546)) < 0.01

In [ ]:
# Why. Close to the minimum but still moving fast, the look-ahead point is
# already on the OTHER SIDE, so its slope has reversed.
beta, lr = 0.9, 0.1
w, v = 0.5, 8.0                 # nearly there, and carrying a lot of speed
look = w - lr * beta * v        # for L = 0.5 w^2 the gradient IS w

print(f"standing at w = {w:+.3f}, velocity v = {v}")
print(f"  gradient here    : {w:+.3f}   -> positive: 'keep moving left'")
print(f"  look-ahead point : {look:+.3f}")
print(f"  gradient there   : {look:+.3f}   -> NEGATIVE: 'you have overshot'")
print("\nMomentum uses the first number and keeps accelerating into the wall.")
print("NAG uses the second and starts braking one step early.")

assert w > 0 > look          # the look-ahead has crossed the minimum

In [ ]:
# It is not always negative - only when the velocity is large enough to
# carry you past. That is exactly when you want the warning.
print(f"{'w':>8}{'v':>8}{'look-ahead':>13}{'slope reversed?':>18}")
for w, v in ((5.0, 8.0), (2.0, 8.0), (0.8, 8.0), (0.5, 8.0), (0.2, 1.0)):
    look = w - lr * beta * v
    print(f"{w:>8.2f}{v:>8.2f}{look:>13.3f}{str(look < 0):>18}")
print("\nFar from the minimum, NAG and momentum agree. The correction only")
print("switches on when overshoot is actually imminent.")

## Part C — On the ravine

In [ ]:
print(f"L = 0.5(x^2 + 20y^2), start (9, 1), lr = {LR}, {STEPS} steps\n")
results = {}
for name, m, n in (("plain GD", 0.0, False), ("momentum", 0.9, False), ("NAG", 0.9, True)):
    path = descend(m, n)
    hit = next((i for i, q in enumerate(path) if loss(q) < 0.1), None)
    results[name] = (loss(path[-1]), hit)
    print(f"  {name:<10} final loss = {loss(path[-1]):8.5f}   "
          f"reached loss<0.1 at step {hit if hit is not None else 'never'}")

assert results["momentum"][1] == 35
assert results["NAG"][1] == 14

## Part D — The result that looks wrong

Read those two lines again. **NAG's final loss is *higher* than momentum's**
(0.00547 against 0.00285) — and NAG is clearly the better optimizer here.

In [ ]:
print(f"momentum: reached the target at step {results['momentum'][1]}, "
      f"final loss {results['momentum'][0]:.5f}")
print(f"NAG     : reached the target at step {results['NAG'][1]}, "
      f"final loss {results['NAG'][0]:.5f}")
print(f"\nNAG arrives {results['momentum'][1] / results['NAG'][1]:.1f}x sooner "
      f"and ends {results['NAG'][0] / results['momentum'][0]:.1f}x higher.")
print("\nBoth are far below the 0.1 threshold and both are still descending.")
print("Comparing final losses at an arbitrary step count measures where each")
print("happened to be when the loop stopped. SPEED OF ARRIVAL is the thing")
print("that matters, and it is what changes the wall-clock time of training.")

In [ ]:
# Check that claim rather than asserting it: give both more steps.
for extra in (60, 120, 240):
    m = loss(descend(0.9, False, steps=extra)[-1])
    n = loss(descend(0.9, True, steps=extra)[-1])
    print(f"  after {extra:>3} steps: momentum {m:.3e}   NAG {n:.3e}")
print("\nThe ordering at any single step count is not a stable ranking.")

## What to take away

- **NAG = momentum with the gradient measured at the look-ahead point**, not the
  current one.
- **`w_look = w_t − ηβv_{t−1}`**, then take the gradient **there**. Only the
  evaluation point changes.
- **Why it damps oscillation:** at the look-ahead point the slope has already
  reversed, so the correction arrives a step early.
- **Momentum cannot see the overshoot coming** — the slope where it stands still
  points forward.
- Measured in 1-D: momentum swings to **−6.04**, NAG to **−3.55** — about
  **41% less overshoot**.
- Measured on the ravine: momentum reached loss < 0.1 at **step 35**, NAG at
  **step 14**.
- NAG's final loss can be slightly *higher* while arriving far sooner — **speed
  of arrival is what matters**.
- **No new hyperparameter, no extra memory, no extra cost** — `nesterov=True`.
- **Still does not fix the single-learning-rate problem.**

## Exercises

1. Sweep β from 0 to 0.99 and plot overshoot for momentum and NAG together. At
   which β does the gap between them open up?
2. NAG's look-ahead uses `ηβv`. Try `ηv` (a full step ahead) instead. Better or
   worse, and why?
3. The classical Nesterov formulation and the one deep-learning frameworks use
   are algebraically different. Look up the Sutskever reparameterisation and
   check numerically that the two agree.
4. Part D argued that final loss is a poor metric here. Design a fairer
   comparison — fixed wall-clock, fixed gradient evaluations, or steps-to-target
   — and rank all three methods under it.
5. Run NAG on lesson 032's saddle point. Does looking ahead help escape it, or
   is this a problem momentum-family methods cannot solve?